dbutils is Databricks’ built-in toolbox, and fs.ls lists the files in a folder, like ls in a terminal.

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/elec_raw")

In [0]:
# test if the load of one works &  connected to pyspark, server and volumes
path = "/Volumes/workspace/default/elec_raw/"
df_18 = spark.read.csv(path + "demanddata_2018.csv")
df_18.printSchema()
display(df_18.limit(5))

Import all files

In [0]:
path = "/Volumes/workspace/default/elec_raw/"
# read in all files
dfs = [] # empty list to hold the 6 DataFrames
for year in range(2018, 2024): # 2018 - 2023 (2024 not included)
    df = spark.read.csv(
        path + f"demanddata_{year}.csv", # e.g. .../elec_raw/
        header=True, # first row = column names
        inferSchema=True # work out number/date types
    )
    dfs.append(df) # add this year to the list

# 2. Stack them into one table, matching columns by name
df_all = dfs[0] # start with 2018
for df in dfs[1:]: # then add 2019-2023 one at a time
    # function - unionByName matches and stacks columnwise
    df_all = df_all.unionByName(df, allowMissingColumns=True)

# 3. Check
print(df_all.count()) # expect 105168
df_all.printSchema()

- For standardisation, make all string col. heads lower case
- save as table in databricks


In [0]:
# returns a new dataframe with newly specified colnames .toDF() - every column at once
# * looks at each col. one at a time and lowers them in case 
df_all = df_all.toDF(*[c.lower() for c in df_all.columns])
# overwrite the current table as a table 
df_all.write.mode("overwrite").saveAsTable("elec_stacked")